# I-3D_web — WebGPU research branch / Colab launcher

**Dedicated `I-3D_web` research branch only (no changes to `main`).** One repository, one set of Python scripts. Colab only clones [Apache0ne/I-3D · I-3D_web](https://github.com/Apache0ne/I-3D/tree/I-3D_web), runs `scripts/colab.py`, and embeds the repository's [3D Studio](https://github.com/Apache0ne/I-3D/tree/main/viewer). **No model installation code or pipeline implementation is duplicated in this notebook.**

Pipeline: BRIA RMBG-2.0 browser alpha removal → image → Hunyuan3D-2mini → Hi3D refinement → Hunyuan3D Paint 2.1 → TokenRig skinning → SOMA 30-joint bridge → START/TARGET 3D poses in the viewer → MotionBricks → animated GLB. Headless Blender is an internal export/rigging dependency, not a separate interface.

**Hardware:** NVIDIA CUDA 13.0 compiler and compatible GPU. A100 40GB is a reasonable development target; high-resolution Hi3D and full PBR generation may still exceed available GPU memory. CUDA-13 Hi3D, Paint and TokenRig installations remain experimental adaptations. **End-to-end inference has not been validated.** Model downloads are substantial, and pretrained weights retain their original licenses. The NVIDIA MotionBricks TensorRT environment retains its own upstream CUDA dependencies.



## 1. Clone the latest `I-3D_web` branch

This branch is intentionally isolated from `main`. The notebook calls the Python code in GitHub; it does not duplicate model implementations.


In [ ]:
from pathlib import Path
import os, sys, subprocess
ROOT = Path("/content/I-3D")
WORK = ROOT / "outputs" / "character"

def run(*args):
    cmd = [str(a) for a in args]
    print("RUN:", " ".join(cmd), flush=True)
    # Inherit stdout and stderr so actual errors are visible in Colab.
    result = subprocess.run(cmd)
    if result.returncode:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(cmd)}")

if not ROOT.exists():
    run("git", "clone", "--branch", "I-3D_web", "--single-branch",
        "https://github.com/Apache0ne/I-3D.git", ROOT)
else:
    # Dedicated experimental branch; never checkout, modify, or commit to main.
    run("git", "-C", ROOT, "fetch", "origin", "I-3D_web")
    run("git", "-C", ROOT, "checkout", "-B", "I-3D_web", "FETCH_HEAD")
run("git", "-C", ROOT, "rev-parse", "--short", "HEAD")
WORK.mkdir(parents=True, exist_ok=True)


## 2. Start the in-Colab I-3D viewer and WebGPU server

The same repository server hosts the interactive 3D Studio, custom WGSL kernel laboratory, and a BRIA RMBG 2.0 background-removal page. Start it once; later stages will reuse it.


In [ ]:
from google.colab import output
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.viewer_server import launch

PORT = 8765
if "_i3d_studio_server" not in globals():
    _i3d_studio_server = launch(WORK, port=PORT)
print("I-3D 3D Studio:")
output.serve_kernel_port_as_iframe(PORT, height=980)


## 3. BRIA RMBG 2.0 — browser-side background removal

**Run this first, before any CUDA model installation.** Select a photograph in the embedded WebGPU page, choose the recommended WebGPU-compatible FP16 model (or the original kn4666 ONNX variant for experimentation), click **Load model and remove background**, and click **Save source + RGBA + mask into I-3D workspace**.

The browser downloads around **513 MB** of model weights. It runs ONNX Runtime Web with the **WebGPU execution provider on your local browser GPU**, not the Colab CUDA GPU. WebGPU failure is reported; the model is never silently replaced with a dummy mask. The patched BRIA model is recommended because the original ONNX export may exceed WebGPU shader binding limits.

**License:** BRIA RMBG 2.0 weights are CC BY-NC 4.0, noncommercial. The code has not yet been executed on a browser GPU as part of this project.


In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(PORT, path="/webgpu/rmbg.html", height=960)
print("BRIA RMBG 2.0 WebGPU stage: export original, mask, and RGBA to the same I-3D workspace.")


## 4. Select the image for subsequent model stages

If you completed the browser BRIA stage and saved all three files, this cell automatically picks the original source and its **verified** transparent cutout. Otherwise, it requests an ordinary local photograph and Hi3D can use its original internal background-removal pipeline.

**Re-run this cell after saving a new browser cutout.**


In [ ]:
from pathlib import Path
import json
from google.colab import files
rmbg_meta=WORK/"00_rmbg_metadata.json"
browser_source=WORK/"00_source_input.png"
browser_cutout=WORK/"00_background_removed.png"
BROWSER_CUTOUT=None
if rmbg_meta.is_file() and browser_source.is_file() and browser_cutout.is_file():
    manifest=json.loads(rmbg_meta.read_text())
    IMAGE=browser_source
    BROWSER_CUTOUT=browser_cutout
    print("Using real BRIA browser cutout for Hi3D:", BROWSER_CUTOUT)
    print("Retaining original RGB image for Paint:", IMAGE)
    print("RMBG source model:", manifest.get("model"))
else:
    uploaded=files.upload()
    if len(uploaded)!=1:
        raise ValueError("Upload exactly one PNG/JPEG/WEBP image")
    filename,data=next(iter(uploaded.items()))
    ext=Path(filename).suffix.lower()
    if ext not in (".png",".jpg",".jpeg",".webp"):
        raise ValueError("Unsupported input format: "+ext)
    IMAGE=WORK/("original_input"+ext)
    IMAGE.write_bytes(data)
    print("No BRIA browser cutout; Hi3D background removal remains enabled.")
print("Original image:",IMAGE)


## 5. Install the remaining I-3D models from the repository

This is a separate heavy CUDA/TensorRT development step for Hi3D, Hunyuan Paint, TokenRig, and MotionBricks. The **BRIA stage above is standalone browser WebGPU**; completing it does not require PyTorch or Colab GPU installation.


In [ ]:
SETUP_STAGE = "all"   # all / hi3d / paint / tokenrig / motion
MOTION_PRECISION = "fp16"  # fp16 or fp32
run(sys.executable, ROOT/"scripts"/"colab.py", "setup",
    "--stage", SETUP_STAGE, "--precision", MOTION_PRECISION)


## WebGPU kernel laboratory — runs on your browser GPU

This **branch-only** addition opens I-3D's custom WGSL compute and rendering pipeline using the same Colab HTTP server. It includes hand-written f32 GEMM, signed INT4 GEMV, LayerNorm/RMSNorm, softmax, convolution, depth-to-points, normal generation, alpha compositing and skinned vertex deformation. Press **Run custom WGSL correctness tests** to compare GPU outputs to independent CPU references.

**The WebGPU device is the GPU in your local Chrome/Edge browser, not Colab's CUDA A100.** The upstream Hi3D/Paint/TokenRig/MotionBricks Python backends remain separate until each complete model graph is ported and its weights adapted.


In [ ]:
from google.colab import output
# Reuse the live I-3D viewer server from the preceding cell.
output.serve_kernel_port_as_iframe(PORT, path="/webgpu/", height=980)
print("WebGPU lab is served directly from branch I-3D_web.")


## 6. Generate textured 3D geometry and rig

The repository CLI uses the BRIA PNG as the geometry reference only when the complete browser metadata matches. Otherwise it takes the original photograph and performs Hi3D internal background removal.


In [ ]:
CHARACTER_FORWARD = "+X"  # +X, -X, +Y, -Y in Blender Z-up
GEOMETRY_RESOLUTION = 512   # 512 / 1024 / 1536
HI3D_STEPS = 25
FORCE_REBUILD = False
args = [sys.executable, ROOT/"scripts"/"colab.py", "prepare",
        "--image", IMAGE, "--forward", CHARACTER_FORWARD,
        "--resolution", str(GEOMETRY_RESOLUTION),
        "--steps", str(HI3D_STEPS), "--output", WORK]
if BROWSER_CUTOUT is not None:
    args.extend(["--cutout", str(BROWSER_CUTOUT)])
if FORCE_REBUILD:
    args.append("--force")
run(*args)
print("Ready to pose:", WORK / "04_soma_character.glb")


## 6. Author two **real 3D poses** in the Studio

In the viewer, select **SOMA 30-joint character**, edit its 3D bones and save **START** and **TARGET** poses. The viewer writes `viewer_poses.json` into the same Colab workspace. The Blender interface is not needed.

Reopen or refresh the Studio iframe below after generating the character; then save both poses before proceeding.


In [ ]:
output.serve_kernel_port_as_iframe(PORT, height=980)
print("Viewer pose file:", WORK/"viewer_poses.json")


## 7. MotionBricks animation and GLB export

The repository scripts convert the two viewer poses into proper Blender keyframes **headlessly**, export pose matrices, generate the 30 FPS MotionBricks transition, then apply it back onto the same textured character. This runs the existing `author_viewer_poses.py`, `export_two_poses.py`, `generate_motion.py` and `apply_motion_blender.py` scripts.


In [ ]:
run(sys.executable, ROOT/"scripts"/"colab.py", "animate",
    "--output", WORK, "--precision", MOTION_PRECISION)
print("Animated GLB:", WORK/"07_animated.glb")
output.serve_kernel_port_as_iframe(PORT, height=980)


## 8. Download all results

The repository collects geometry, PBR model, skinning, SOMA character, pose files, MotionBricks animation, diagnostics and editable `.blend` into one ZIP. No second viewer is needed.


In [ ]:
run(sys.executable, ROOT/"scripts"/"colab.py", "pack", "--output", WORK)
from google.colab import files
files.download(str(WORK/"I3D_output_bundle.zip"))


## Sources and limitations

- [All I-3D scripts](https://github.com/Apache0ne/I-3D/tree/I-3D_web/scripts) — installers, orchestration, 3D rigging, animation and packaging.
- [Viewer source](https://github.com/Apache0ne/I-3D/tree/I-3D_web/viewer) — interactive Colab Studio.
- [Hi3D original implementation](https://github.com/hi3d-official/Hi3Dv1.0), [Hunyuan3D-2](https://github.com/Tencent-Hunyuan/Hunyuan3D-2), [Hunyuan Paint 2.1](https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1), [SkinTokens](https://github.com/VAST-AI-Research/SkinTokens), [SOMA-X](https://github.com/NVlabs/SOMA-X), [MotionBricks](https://github.com/NVlabs/motionbricks).

This simplified notebook is a **launcher, not a validated single-click production model**. Hi3D's upstream reference stack was CUDA 12.4; our CUDA-13 SpConv and FlashAttention wheels are unofficial, checksum-verified builds. Skinning quality, external model weights and TensorRT compatibility require actual inference on the chosen Colab runtime. Upstream code and model licenses apply independently.
BRIA source [kn4666/RMBG 2.0 web](https://huggingface.co/kn4666/bria-rmbg-2.0-web) and [WebGPU-patched RMBG 2.0](https://huggingface.co/yamura4/RMBG-2.0-WebGPU) are third-party noncommercial BRIA weights. The browser stage uses ONNX Runtime Web; its complete model operator graph is **not** composed of custom I-3D WGSL shaders.\n